# 15 · Blind equalization: CMA pre-convergence, then RDE

The equalizers of [14](14-adaptive-equalization.ipynb) need known symbols to start.  A
**blind** equalizer needs none: it adapts the taps to restore a property
every transmitted symbol has.

* The **constant modulus algorithm** (CMA) pulls every output toward one
  circle, `|y|² = R`: exact for PSK and a usable approximation for QAM.
  It ignores the phase, so it works under laser phase noise and leaves the
  carrier to a later CPR stage.
* The **radius-directed equalizer** (RDE) pulls each output to the
  *nearest* ring of the constellation (three for 16-QAM).

On 16-QAM, RDE reaches a lower steady-state error than CMA because it
targets each symbol's own ring instead of one Godard circle.  From a cold
start, though, RDE converges several times slower: with the taps far from
the solution the nearest ring is often the wrong one.  The usual receiver
starts with CMA and switches to RDE once CMA has opened the eye.

In CommKit the switch is one argument: `block_rde(..., state=cma_result.state)`
continues where `block_cma` stopped, with its taps, its pending input and its
normalization.  The same works for the per-symbol `cma` and `rde`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import dispatch
from commkit.recovery import BPS

# Build the data on the GPU when there is one; every later step follows it.
DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()

## 1. Link

Dual-polarization 16-QAM at 32 GBd and 2 samples per symbol, a 30 degree
polarization rotation, 100 kHz laser phase noise from one laser and AWGN at
Es/N0 = 20 dB.

In [ ]:
C16 = Constellation.qam(16)
tx = ck.generate(
    C16,
    num_symbols=2**17,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    num_channels=2,
    rng=1,
    device=DEVICE,
)
rx = ck.impairments.apply_polarization_mixing(tx, theta=np.deg2rad(30))
rx = ck.impairments.apply_phase_noise(rx, linewidth=100e3, shared_lo=True, rng=2)
rx = ck.impairments.apply_awgn(rx, esn0_db=20, rng=3)
rx = ck.filtering.matched_filter(rx)
ref = rx.reference.symbols

A blind equalizer leaves the carrier phase to a later stage, and both CMA and
RDE are blind to a rotation by a multiple of 90 degrees and to which output
is which polarization.  To score the output, this helper runs blind phase
search, resolves that ambiguity against the reference in each window, and
returns the symbol error rate per window of 8192 symbols.

In [ ]:
WINDOW = 8192


def ser_per_window(y):
    y = ck.recovery.correct_carrier_phase(
        y, BPS(), constellation=C16, sampling_rate=rx.symbol_rate
    )
    ser = []
    for k in range(0, y.shape[-1] - WINDOW + 1, WINDOW):
        yk, rk = y[..., k : k + WINDOW], ref[..., k : k + WINDOW]
        yk = ck.recovery.resolve_phase_ambiguity(yk, rk, constellation=C16)
        ser.append(ck.metrics.ser(yk, rk, constellation=C16).mean())
    return np.array(ser)

## 2. RDE from a cold start

Both equalizers run with their default step sizes (see the last section).

In [ ]:
B = 2048
cold = ck.equalization.block_rde(rx, num_taps=21, block_size=B)
ser_cold = ser_per_window(cold.y_hat)

## 3. CMA first, then RDE with its state

CMA runs over the first 16 blocks; RDE continues on the rest of the record
from `pre.state`.  The two outputs join at `pre.state.overlap`: the last
symbols of the first call are computed again by the second, which sees their
full tap window.

In [ ]:
x = rx.samples
n_pre = 16 * B  # symbols for CMA
kw = dict(sps=2, constellation=C16, num_taps=21, block_size=B)

pre = ck.equalization.block_cma(x[..., : 2 * n_pre], **kw)
post = ck.equalization.block_rde(x[..., 2 * n_pre :], state=pre.state, **kw)

xp = dispatch(x)[1]
keep = pre.y_hat.shape[-1] - pre.state.overlap
y_handover = xp.concatenate([pre.y_hat[..., :keep], post.y_hat], axis=-1)
ser_handover = ser_per_window(y_handover)

# For comparison: CMA over the whole record.
cma_only = ck.equalization.block_cma(x, **kw)
ser_cma = ser_per_window(cma_only.y_hat)

In [ ]:
k = (np.arange(ser_cold.size) + 0.5) * WINDOW
floor = 1 / (2 * WINDOW)  # one error in both polarizations
fig, ax = plt.subplots()
ax.semilogy(k, np.maximum(ser_cold, floor), "o-", label="RDE, cold start")
ax.semilogy(k, np.maximum(ser_cma, floor), "s-", label="CMA only")
ax.semilogy(k, np.maximum(ser_handover, floor), "^-", label="CMA, then RDE")
ax.axvline(n_pre, color="gray", ls=":", label="handover")
ax.set_xlabel("symbol")
ax.set_ylabel(f"SER per {WINDOW} symbols")
ax.legend()
plt.show()

RDE from a cold start needs most of the record before it opens the eye.
With CMA first, the handover point arrives with the eye already open, and
RDE goes on from there.

## 4. Steady state

Over the last 32768 symbols, after both have converged, RDE's ring targets
give a lower error vector magnitude than CMA's single circle.

In [ ]:
tail = slice(-(2**15), None)
for name, y in [("CMA only", cma_only.y_hat), ("CMA, then RDE", y_handover)]:
    z = ck.recovery.correct_carrier_phase(
        y, BPS(), constellation=C16, sampling_rate=rx.symbol_rate
    )
    z = ck.recovery.resolve_phase_ambiguity(
        z[..., tail], ref[..., tail], constellation=C16
    )
    print(f"{name:14s} EVM per polarization:", ck.metrics.evm(z, ref[..., tail]))

## Choosing the steps

* The block equalizers hold their taps across a block of `block_size`
  symbols, so the stable `step_size` falls as `1/block_size`: keep
  `step_size * block_size` below about 0.5 (CMA) or 1 (RDE).  The defaults
  are a fixed per-symbol step (2e-4 for CMA, 1e-3 for RDE), capped at
  `0.5 / block_size` for RDE.
* 64-QAM RDE does not converge from a cold start at any step; it needs CMA
  first.
* The per-symbol `cma` and `rde` take the same two-call pattern:
  `rde(x_rest, state=cma(x_first).state)`.  They adapt every symbol, so
  they converge in fewer symbols and run faster on the CPU; the block
  versions are for the GPU.
* `initial_taps=pre.weights` also seeds RDE with the CMA taps, but starts a
  new record: it pads the start again and recomputes the normalization.
  `state=` continues the same record.

**Next:** [16 · Polarization multiplexing](16-polarization-multiplexing.ipynb).